In [21]:
from pyspark.sql import functions as F

crm_df = spark.table("lh_telecom_silver.dbo.silver_crm_customer")
billing_df = spark.table("lh_telecom_silver.dbo.silver_billing_churn")
service_df = spark.table("lh_telecom_silver.dbo.silver_service_provisioning")

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 23, Finished, Available, Finished, False)

In [22]:
def check_customer_uniqueness(df, table_name):
    result = (
        df.agg(
            F.count("*").alias("row_count"),
            F.countDistinct("customer_id").alias("distinct_customer_count"),
            F.sum(
                F.when(F.col("customer_id").isNull(), 1).otherwise(0)
            ).alias("null_customer_ids")
        )
    )

    print(table_name)
    display(result)


check_customer_uniqueness(crm_df, "CRM")
check_customer_uniqueness(billing_df, "Billing")
check_customer_uniqueness(service_df, "Service")

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 24, Finished, Available, Finished, False)

CRM


SynapseWidget(Synapse.DataFrame, 4231ebc1-4314-4343-872c-1d5e0ca4aeca)

Billing


SynapseWidget(Synapse.DataFrame, e56837be-3900-47b2-a270-3217207e035e)

Service


SynapseWidget(Synapse.DataFrame, 87775387-c8db-46d4-8fdc-797b56efba36)

In [23]:
crm_ids = crm_df.select("customer_id")
billing_ids = billing_df.select("customer_id")
service_ids = service_df.select("customer_id")

print("CRM customers:", crm_ids.distinct().count())
print("Billing customers:", billing_ids.distinct().count())
print("Service customers:", service_ids.distinct().count())

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 25, Finished, Available, Finished, False)

CRM customers: 6857
Billing customers: 6857
Service customers: 6857


In [24]:
crm_missing_billing = crm_ids.join(
    billing_ids,
    on="customer_id",
    how="left_anti"
)

crm_missing_service = crm_ids.join(
    service_ids,
    on="customer_id",
    how="left_anti"
)

print(
    "CRM customers missing from Billing:",
    crm_missing_billing.count()
)

print(
    "CRM customers missing from Service:",
    crm_missing_service.count()
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 26, Finished, Available, Finished, False)

CRM customers missing from Billing: 0
CRM customers missing from Service: 0


In [25]:
ml_base_df = (
    crm_df.alias("crm")
    .join(
        billing_df.alias("billing"),
        on="customer_id",
        how="inner"
    )
    .join(
        service_df.alias("service"),
        on="customer_id",
        how="inner"
    )
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 27, Finished, Available, Finished, False)

In [26]:
print("CRM rows:", crm_df.count())
print("Joined rows:", ml_base_df.count())
print("Joined columns:", len(ml_base_df.columns))

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 28, Finished, Available, Finished, False)

CRM rows: 6857
Joined rows: 6857
Joined columns: 29


In [27]:
for column_name in sorted(ml_base_df.columns):
    print(column_name)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 29, Finished, Available, Finished, False)

churn_label
churn_reason
churn_value
city
contract
country
customer_id
dependent
device_protection
gender
internet_service
latitude
longitude
monthly_charges
multiple_lines
online_backup
online_security
paperless_billing
partner
payment_method
phone_service
senior_citizen
state
streaming_movies
streaming_tv
tech_support
tenure_months
total_charges
zip_code


In [28]:
from collections import Counter

column_counts = Counter(ml_base_df.columns)

duplicate_columns = [
    column_name
    for column_name, count in column_counts.items()
    if count > 1
]

print("Duplicate columns:", duplicate_columns)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 30, Finished, Available, Finished, False)

Duplicate columns: []


In [29]:
leakage_columns = [
    "churn_label",
    "churn_reason",
]

existing_leakage_columns = [
    column_name
    for column_name in leakage_columns
    if column_name in ml_base_df.columns
]

ml_base_df = ml_base_df.drop(*existing_leakage_columns)

print("Removed leakage columns:", existing_leakage_columns)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 31, Finished, Available, Finished, False)

Removed leakage columns: ['churn_label', 'churn_reason']


In [30]:
required_columns = [
    # Identifier
    "customer_id",

    # Demographics
    "gender",
    "senior_citizen",
    "partner",
    "dependent",

    # Account and billing
    "tenure_months",
    "contract",
    "paperless_billing",
    "payment_method",
    "monthly_charges",
    "total_charges",

    # Telecom services
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",

    # Target
    "churn_value"
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in ml_base_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

ml_selected_df = ml_base_df.select(
    *required_columns
)

print(
    f"Selected {len(ml_selected_df.columns)} columns."
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 32, Finished, Available, Finished, False)

Selected 21 columns.


In [31]:
categorical_columns = [
    "gender",
    "partner",
    "senior_citizen",
    "dependent",
    "contract",
    "paperless_billing",
    "payment_method",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

ml_clean_df = ml_selected_df

for column_name in categorical_columns:
    ml_clean_df = ml_clean_df.withColumn(
        column_name,
        F.trim(F.col(column_name).cast("string"))
    )

ml_clean_df = (
    ml_clean_df
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "tenure_months",
        F.col("tenure_months").cast("integer")
    )
    .withColumn(
        "monthly_charges",
        F.col("monthly_charges").cast("double")
    )
    .withColumn(
        "total_charges",
        F.col("total_charges").cast("double")
    )
    .withColumn(
        "churn_value",
        F.col("churn_value").cast("integer")
    )
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 33, Finished, Available, Finished, False)

In [32]:
def yes_indicator(column_name: str):
    return (
        F.when(
            F.lower(
                F.trim(F.col(column_name))
            ) == "yes",
            F.lit(1)
        )
        .otherwise(F.lit(0))
    )

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 34, Finished, Available, Finished, False)

In [33]:
ml_features_df = (
    ml_clean_df

    # 1. Customer has internet service
    .withColumn(
        "has_internet_service",
        F.when(
            F.lower(F.col("internet_service"))
            .isin(
                "no",
                "none",
                "no internet service"
            ),
            0
        ).otherwise(1)
    )

    # 2. Customer has multiple phone lines
    .withColumn(
        "has_multiple_lines",
        yes_indicator("multiple_lines")
    )

    # 3. Security bundle components
    .withColumn(
        "has_online_security",
        yes_indicator("online_security")
    )
    .withColumn(
        "has_online_backup",
        yes_indicator("online_backup")
    )
    .withColumn(
        "has_device_protection",
        yes_indicator("device_protection")
    )
    .withColumn(
        "has_tech_support",
        yes_indicator("tech_support")
    )

    # 4. Streaming bundle components
    .withColumn(
        "has_streaming_tv",
        yes_indicator("streaming_tv")
    )
    .withColumn(
        "has_streaming_movies",
        yes_indicator("streaming_movies")
    )

    # 5. Number of security-related services
    .withColumn(
        "security_service_count",
        F.col("has_online_security")
        + F.col("has_online_backup")
        + F.col("has_device_protection")
        + F.col("has_tech_support")
    )

    # 6. Complete security bundle indicator
    .withColumn(
        "has_security_bundle",
        F.when(
            (
                F.col("has_online_security")
                + F.col("has_device_protection")
                + F.col("has_tech_support")
            ) >= 2,
            1
        ).otherwise(0)
    )

    # 7. Number of streaming services
    .withColumn(
        "streaming_service_count",
        F.col("has_streaming_tv")
        + F.col("has_streaming_movies")
    )

    # 8. Streaming bundle indicator
    .withColumn(
        "has_streaming_bundle",
        F.when(
            F.col("streaming_service_count") == 2,
            1
        ).otherwise(0)
    )

    # 9. Total number of subscribed services
    .withColumn(
        "total_service_count",
        yes_indicator("phone_service")
        + F.col("has_multiple_lines")
        + F.col("has_internet_service")
        + F.col("has_online_security")
        + F.col("has_online_backup")
        + F.col("has_device_protection")
        + F.col("has_tech_support")
        + F.col("has_streaming_tv")
        + F.col("has_streaming_movies")
    )

    # 10. Contract duration represented numerically
    .withColumn(
        "contract_length_years",
        F.when(
            F.lower(F.col("contract"))
            .isin(
                "month-to-month",
                "month to month"
            ),
            F.lit(0)
        )
        .when(
            F.lower(F.col("contract"))
            .isin(
                "one year",
                "1 year"
            ),
            F.lit(1)
        )
        .when(
            F.lower(F.col("contract"))
            .isin(
                "two year",
                "two years",
                "2 year",
                "2 years"
            ),
            F.lit(2)
        )
        .otherwise(F.lit(None).cast("integer"))
    )

    # 11. Long-term contract indicator
    .withColumn(
        "has_long_term_contract",
        F.when(
            F.col("contract_length_years") >= 1,
            1
        ).otherwise(0)
    )

    # 12. Month-to-month contract risk indicator
    .withColumn(
        "is_month_to_month",
        F.when(
            F.col("contract_length_years") == 0,
            1
        ).otherwise(0)
    )

    # 13. Average historical monthly charge
    .withColumn(
        "average_historical_monthly_charge",
        F.when(
            F.col("tenure_months") > 0,
            F.col("total_charges")
            / F.col("tenure_months")
        ).otherwise(F.col("monthly_charges"))
    )

    # 14. Difference between current and historical charge
    .withColumn(
        "current_vs_historical_charge_difference",
        F.col("monthly_charges")
        - F.col("average_historical_monthly_charge")
    )

    # 15. Charge per subscribed service
    .withColumn(
        "monthly_charge_per_service",
        F.when(
            F.col("total_service_count") > 0,
            F.col("monthly_charges")
            / F.col("total_service_count")
        ).otherwise(F.col("monthly_charges"))
    )

    # 16. Tenure group
    .withColumn(
        "tenure_group",
        F.when(
            F.col("tenure_months") <= 6,
            "0-6 Months"
        )
        .when(
            F.col("tenure_months") <= 12,
            "7-12 Months"
        )
        .when(
            F.col("tenure_months") <= 24,
            "13-24 Months"
        )
        .when(
            F.col("tenure_months") <= 48,
            "25-48 Months"
        )
        .otherwise("49+ Months")
    )

    # 17. New-customer indicator
    .withColumn(
        "is_new_customer",
        F.when(
            F.col("tenure_months") <= 6,
            1
        ).otherwise(0)
    )

    # 18. Automatic-payment indicator
    .withColumn(
        "uses_automatic_payment",
        F.when(
            F.lower(F.col("payment_method"))
            .contains("automatic"),
            1
        ).otherwise(0)
    )

    # 19. Electronic-check indicator
    .withColumn(
        "uses_electronic_check",
        F.when(
            F.lower(F.col("payment_method"))
            .contains("electronic check"),
            1
        ).otherwise(0)
    )

    # 20. Paperless billing indicator
    .withColumn(
        "uses_paperless_billing",
        yes_indicator("paperless_billing")
    )
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 35, Finished, Available, Finished, False)

In [34]:
target_validation_df = (
    ml_features_df
    .agg(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(
                F.col("churn_value").isNull(),
                1
            ).otherwise(0)
        ).alias("null_targets"),
        F.sum(
            F.when(
                ~F.col("churn_value").isin(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_targets")
    )
)

display(target_validation_df)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c08901d3-2c4f-4062-84c6-88d2d26a5c37)

In [35]:
target_validation = (
    target_validation_df
    .collect()[0]
)

if (
    target_validation["null_targets"] > 0
    or target_validation["invalid_targets"] > 0
):
    raise ValueError(
        "The target column contains null or invalid values."
    )

print("Target validation passed.")

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 37, Finished, Available, Finished, False)

Target validation passed.


In [36]:
total_customers = ml_features_df.count()

class_distribution_df = (
    ml_features_df
    .groupBy("churn_value")
    .count()
    .withColumnRenamed(
        "count",
        "customer_count"
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("customer_count")
            / F.lit(total_customers)
            * 100,
            2
        )
    )
    .orderBy("churn_value")
)

display(class_distribution_df)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 98cd060a-d96c-4817-b042-f6b555e91267)

In [37]:
important_columns = [
    "customer_id",
    "senior_citizen",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "contract",
    "payment_method",
    "churn_value"
]

null_summary_expressions = [
    F.sum(
        F.when(
            F.col(column_name).isNull(),
            1
        ).otherwise(0)
    ).alias(column_name)
    for column_name in important_columns
]

null_summary_df = ml_features_df.agg(
    *null_summary_expressions
)

display(null_summary_df)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9556292a-586d-4452-b4d0-0d8981052897)

In [38]:
ml_features_df = (
    ml_features_df
    .filter(
        F.col("tenure_months") >= 0
    )
    .filter(
        F.col("monthly_charges") >= 0
    )
    .filter(
        F.col("total_charges") >= 0
    )
    .filter(
        F.col("churn_value").isin(0, 1)
    )
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 40, Finished, Available, Finished, False)

In [39]:
final_columns = [
    # Identifier
    "customer_id",

    # Original demographic features
    "gender",
    "senior_citizen",
    "partner",
    "dependent",

    # Original account and billing features
    "tenure_months",
    "contract",
    "paperless_billing",
    "payment_method",
    "monthly_charges",
    "total_charges",

    # Original service features
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",

    # Engineered service features
    "has_internet_service",
    "has_multiple_lines",
    "has_online_security",
    "has_online_backup",
    "has_device_protection",
    "has_tech_support",
    "has_streaming_tv",
    "has_streaming_movies",
    "security_service_count",
    "has_security_bundle",
    "streaming_service_count",
    "has_streaming_bundle",
    "total_service_count",

    # Engineered contract features
    "contract_length_years",
    "has_long_term_contract",
    "is_month_to_month",

    # Engineered financial features
    "average_historical_monthly_charge",
    "current_vs_historical_charge_difference",
    "monthly_charge_per_service",

    # Engineered customer behaviour features
    "tenure_group",
    "is_new_customer",
    "uses_automatic_payment",
    "uses_electronic_check",
    "uses_paperless_billing",

    # Target
    "churn_value"
]

ml_final_df = ml_features_df.select(
    *final_columns
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 41, Finished, Available, Finished, False)

In [40]:
final_row_count = ml_final_df.count()
final_customer_count = (
    ml_final_df
    .select("customer_id")
    .distinct()
    .count()
)

print(f"Final rows: {final_row_count:,}")
print(
    f"Distinct customers: "
    f"{final_customer_count:,}"
)
print(
    f"Final columns: "
    f"{len(ml_final_df.columns)}"
)

if final_row_count != final_customer_count:
    raise ValueError(
        "The final ML feature dataset does not contain "
        "exactly one row per customer."
    )

display(ml_final_df.limit(10))

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 42, Finished, Available, Finished, False)

Final rows: 6,857
Distinct customers: 6,857
Final columns: 45


SynapseWidget(Synapse.DataFrame, 6762edf7-58a7-4909-8cb4-f5b552bc1127)

In [41]:
GOLD_ML_TABLE = (
    "lh_telecom_gold.dbo.ml_customer_churn_features"
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 43, Finished, Available, Finished, False)

In [42]:
(
    ml_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ML_TABLE)
)

print(
    f"Saved Gold table: {GOLD_ML_TABLE}"
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 44, Finished, Available, Finished, False)

Saved Gold table: lh_telecom_gold.dbo.ml_customer_churn_features


In [44]:
PARQUET_OUTPUT_PATH = (
    "Files/machine_learning/customer_churn_features"
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 46, Finished, Available, Finished, False)

In [45]:
(
    ml_final_df
    .coalesce(1)
    .write
    .format("parquet")
    .mode("overwrite")
    .save(PARQUET_OUTPUT_PATH)
)

print(
    "Exported Azure ML input files to:",
    PARQUET_OUTPUT_PATH
)

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 47, Finished, Available, Finished, False)

Exported Azure ML input files to: Files/machine_learning/customer_churn_features


In [46]:
saved_gold_df = spark.table(
    GOLD_ML_TABLE
)

saved_parquet_df = spark.read.parquet(
    PARQUET_OUTPUT_PATH
)

print(
    "Gold table rows:",
    saved_gold_df.count()
)

print(
    "Parquet rows:",
    saved_parquet_df.count()
)

print(
    "Gold table columns:",
    len(saved_gold_df.columns)
)

print(
    "Parquet columns:",
    len(saved_parquet_df.columns)
)

if saved_gold_df.count() != saved_parquet_df.count():
    raise ValueError(
        "Gold Delta and Parquet output row counts do not match."
    )

display(saved_gold_df.limit(5))

StatementMeta(, 000edf3d-7177-4e83-86d1-eb3f2bf43424, 48, Finished, Available, Finished, False)

Gold table rows: 6857
Parquet rows: 6857
Gold table columns: 45
Parquet columns: 45


SynapseWidget(Synapse.DataFrame, d12806f6-cb57-4500-8af4-e84db3a73fd8)